# Clase 1 — Una función es una máquina con perillas

**Deep Learning para Cientistas Sociales**

> **La idea de hoy en una línea.** Una función es una máquina: le metés algo, sale
> algo, y tiene perillas al costado. Cambiar lo que le metés te *recorre* la figura;
> mover las perillas *deforma* la figura. Un modelo es la figura. Entrenar es girar
> las perillas.

---

**Cuando termines este notebook vas a poder:**

1. Escribir una función en Python, graficarla y anticipar qué le pasa al dibujo si
   cambiás un número.
2. Explicar con tus palabras la diferencia entre una **variable** y un **parámetro**.
3. Ajustar a ojo una recta a datos reales y leer un número que dice qué tan mal está.
4. Traducir un parámetro a una **afirmación sustantiva en castellano** (que es la
   parte que va en tu paper).

**Lo que hace falta traer:** nada. Es la primera clase.

---

### Cómo se usa esto (si nunca abriste un notebook, leé estas seis líneas)

- Un notebook son **celdas**. Algunas tienen texto (como esta) y otras tienen código.
- Para correr una celda: hacé clic adentro y apretá **`Shift + Enter`**. También podés
  apretar el ▶ que aparece a la izquierda cuando pasás el mouse.
- Corré las celdas **en orden, de arriba hacia abajo**. Si una celda usa algo que
  definiste más arriba y esa de más arriba no la corriste, se queja. Es el error
  número uno y le pasa a todo el mundo.
- Donde dice `# TODO` te toca a vos. Cambiá números, borrá cosas, probá.
- **Si rompés algo no pasa nada.** En serio: nada. `Entorno → Reiniciar` y volvés a
  correr desde arriba. No hay forma de romper tu computadora desde acá.
- Un error rojo enorme **no es un reto**: es información. Casi siempre la última
  línea dice qué pasó.

## 0. Preparar el entorno

Esta celda es **idéntica en los diez notebooks del curso**. En Colab clona el
repo y deja todo a mano; en tu máquina no hace prácticamente nada porque ya
tenés los archivos. Corrémela y seguí de largo: no hay nada que entender acá.

In [ ]:
# =============================================================================
# BOOTSTRAP — la misma celda en todos los notebooks del curso. No la edites.
# En Colab: clona el repo del curso (o usa el .zip que subas) e instala lo que falte.
# En tu máquina: encuentra el repo subiendo carpetas y no instala nada.
# =============================================================================
import os
import pathlib
import subprocess
import sys
import zipfile

REPO = "https://github.com/johannperez/dlss"
RAMA = "main"
CARPETA = "dlss"


def en_colab():
    """¿Estamos en Google Colab? Se lo preguntamos a Python, no lo adivinamos."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def buscar_raiz(desde=None):
    """Sube carpeta por carpeta hasta encontrar la que tiene `_shared/ssdl`."""
    punto = pathlib.Path(desde or os.getcwd()).resolve()
    for candidata in [punto, *punto.parents]:
        if (candidata / "_shared" / "ssdl").is_dir():
            return candidata
    return None


def raiz_desde_zip():
    """Plan B: si subiste el .zip del repo (dlss-main.zip o parecido), lo abre."""
    for archivo in sorted(pathlib.Path(".").glob("dlss*.zip")):
        with zipfile.ZipFile(archivo) as z:
            z.extractall(CARPETA + "-zip")
        for ssdl in pathlib.Path(CARPETA + "-zip").glob("**/_shared/ssdl"):
            return ssdl.parent.parent.resolve()
    return None


raiz = buscar_raiz()

if raiz is None and en_colab():
    if not pathlib.Path(CARPETA).exists():
        print("Bajando el repo del curso (una sola vez por sesión)...")
        clon = subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                               REPO, CARPETA],
                              env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
        if clon.returncode != 0:
            print("No pude clonar el repo. Pruebo con un .zip subido a mano...")
    raiz = buscar_raiz(CARPETA) if pathlib.Path(CARPETA).exists() else None
    raiz = raiz or raiz_desde_zip()

if raiz is None:
    raise SystemExit(
        "No encuentro el repo del curso.\n"
        "· En Colab, si el clone falló (por ejemplo, «Repository not found» o 404):\n"
        "  el repo github.com/johannperez/dlss tiene que ser público. Si no lo es,\n"
        "  o no hay red hacia GitHub, conseguí el .zip del repo (en la página del\n"
        "  repo: Code → Download ZIP; o pedíselo a quien da el curso), subilo con\n"
        "  el panel de archivos de Colab (el nombre tiene que empezar con `dlss`,\n"
        "  como dlss-main.zip) y volvé a correr esta celda.\n"
        "· En tu máquina: abrí este notebook desde adentro de la carpeta `dlss`.\n"
        "Más detalle en recursos/setup.md."
    )

# Dependencias. En Colab casi todas vienen puestas: instalamos sólo lo que falte.
# En tu máquina no instalamos nada (torch es pesado y varias clases no lo usan):
# si falta algo, avisamos y lo instalás vos (ver recursos/setup.md).
faltan = []
for modulo, paquete in [("numpy", "numpy"), ("matplotlib", "matplotlib"),
                        ("torch", "torch")]:
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan and en_colab():
    print(f"Instalando {', '.join(faltan)}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan],
                   check=True)
elif faltan:
    print(f"[!] Falta {', '.join(faltan)}. Si este notebook lo usa, instalalo con:\n"
          f"    {sys.executable} -m pip install {' '.join(faltan)}")

sys.path.insert(0, str(raiz / "_shared"))   # para poder hacer `import ssdl`
os.environ["DLSS_RAIZ"] = str(raiz)         # para que ssdl encuentre los datos

print(f"Listo. Repo del curso en: {raiz}")

### Los imports y la semilla

`ssdl` es el paquete del curso: la plomería que no queremos repetir en diez notebooks
(traer los datos, dibujar con los colores de siempre). No esconde nada importante,
está en `_shared/ssdl/` y se puede leer.

Fijamos la **semilla** en 1983 siempre. No hace al modelo mejor: hace a tu resultado
reproducible, que es condición para poder afirmar algo. Desde la clase 4 explicamos
por qué; desde hoy lo hacemos.

Si esta celda imprime **«Todo listo ✓»**, ya está: no hay nada más que configurar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ssdl import datos, graficos as g, utils

utils.fijar_semilla(1983)          # la semilla del curso

print("Todo listo ✓")
print(f"numpy {np.__version__}")

Y una función de la casa, que vamos a usar para revisar los ejercicios. **No corrige
como un examen**: te dice qué le falta a tu respuesta y te deja seguir.

In [ ]:
def revisar(condicion, bien, ayuda):
    """Revisa un ejercicio con buenos modales.

    No levanta excepciones: si te falta algo te lo dice y seguís. Un `assert`
    pelado te tira treinta líneas rojas y no te explica nada, y eso no ayuda
    a nadie a última hora.
    """
    if condicion:
        print("✓ " + bien)
    else:
        print("Todavía no — " + ayuda)

---

## 1. La idea — la caja con perillas

Una función es una máquina:

```
                        a            b
                        │            │      <-- las PERILLAS (parámetros)
                    ┌───┴────────────┴───┐
        x  ───────► │     f(x) = a·x + b │ ───────►  f(x)
   la VARIABLE      └────────────────────┘        lo que sale
   (lo que entra)
```

Dos aclaraciones que ahorran media hora de confusión:

- **`f(x)` no se lee «f por x».** No es una multiplicación. Son los paréntesis de
  «aplicá f a x». Es una notación desafortunada y la sufrimos todos.
- **`x` y `a` son las dos letras, pero no son la misma clase de cosa.** `x` cambia
  de un caso a otro (el año 1998, el año 1999…). `a` es la misma para todos los casos:
  la elegís vos.

**La analogía de tu propia investigación.** En una encuesta, las *respuestas* cambian
de persona a persona: son variables (cada una es una columna de tu base). El
*coeficiente* de tu regresión es uno solo para toda la muestra: es un parámetro. Nadie
confunde la edad de un encuestado con el coeficiente de la edad. Es exactamente la
misma distinción.

(Ojo con el **peso muestral**: suena a parámetro, pero cada encuestado tiene el suyo y
vive en una columna. Por la regla del dedo, es una variable.)

**Lo que NO es.** La máquina no descubre la fórmula. **Vos elegís la familia de
figuras** (una recta, una S, una red neuronal) y la máquina elige un miembro de esa
familia. La elección de la familia es tuya y es teórica.

### Escribamos la caja en Python

Esta es la misma función que está en las slides. Leela línea por línea:

- `def` quiere decir «voy a definir una función». Es una palabra de Python, no una
  abreviatura misteriosa.
- `recta` es el nombre que le ponemos. Podría ser cualquiera.
- `(x, a, b)` son las tres cosas que la función recibe.
- **Los dos puntos del final son obligatorios.** Y lo que va adentro de la función
  va **indentado** (corrido a la derecha). Python es maniático con los espacios y
  no es culpa tuya.
- `return` es lo que sale por la boca de salida.

In [ ]:
def recta(x, a, b):
    """La máquina: entra x, salen a*x + b. `a` y `b` son las perillas."""
    return a * x + b


# Probemos la máquina metiéndole tres cosas distintas, con las MISMAS perillas.
print("con a=2, b=5:")
print("  si entra x=0  -> sale", recta(0, a=2, b=5))
print("  si entra x=1  -> sale", recta(1, a=2, b=5))
print("  si entra x=10 -> sale", recta(10, a=2, b=5))

Fijate lo que acabás de hacer: **moviste `x` y las perillas quedaron quietas**.
Estabas recorriendo la misma figura, mirando distintos casos de la misma teoría.

Ahora al revés: dejamos el caso quieto (`x = 10`) y movemos las perillas.

In [ ]:
print("el mismo caso (x=10), con tres teorías distintas:")
print("  a=2,   b=5   ->", recta(10, a=2, b=5))
print("  a=0.5, b=5   ->", recta(10, a=0.5, b=5))
print("  a=2,   b=50  ->", recta(10, a=2, b=50))

---

## 2. Miralo — la figura que dibuja la máquina

Hasta acá vimos números sueltos. Una función es más fácil de entender **dibujada**:
le pedimos el valor en muchos puntos seguidos y unimos.

`np.linspace(0, 10, 100)` quiere decir «dame 100 números repartidos parejos entre
0 y 10». Es la forma de decir «recorré el eje x».

> **Apostá primero.** Antes de correr la celda: la recta que va a salir, ¿por dónde
> va a pasar cuando x valga 0? Sin la predicción fallida, un gráfico lindo es sólo
> un gráfico lindo.

In [ ]:
xs = np.linspace(0, 10, 100)          # 100 valores de x, de 0 a 10

fig, ax = g.figura(7, 4.2, titulo="Mi primera función dibujada",
                   xlabel="x  (la variable)", ylabel="f(x)")
ax.plot(xs, recta(xs, a=2, b=5), color=g.MODELO, linewidth=3)
plt.show()

### La celda de juego

Esta celda es para romperla. Cambiá los dos números de arriba y volvé a correrla.
**Corrémela cinco veces con números distintos** antes de seguir: una con `a` negativo,
una con `a = 0`, una con `b` muy grande.

Después de cada corrida, preguntate: *¿la figura giró o se movió entera?*

In [ ]:
a = 2.0      # ← cambiá este número. La INCLINACIÓN: cuánto sube por cada paso de x.
b = 5.0      # ← cambiá este también. La ALTURA: cuánto vale cuando x es 0.

fig, ax = g.figura(7, 4.2, titulo=f"f(x) = {a}·x + {b}",
                   xlabel="x", ylabel="f(x)")
ax.set_ylim(-20, 60)                       # el encuadre fijo, para que se note el cambio
ax.plot(xs, recta(xs, a, b), color=g.MODELO, linewidth=3)
ax.scatter([0], [recta(0, a, b)], s=90, color=g.PARAMETRO, zorder=5)
ax.annotate(f"acá arranca: {recta(0, a, b):.1f}", (0, recta(0, a, b)),
            textcoords="offset points", xytext=(12, -26), color=g.PARAMETRO,
            fontweight="bold")
plt.show()

print(f"En castellano: cuando x es 0 vale {b}, y de ahí sube {a} por cada paso de x.")

### `TODO 1` — dibujá una recta a pedido

Escribí los dos números que hacen que la recta **pase por (0, 10)** y **baje 2 por
cada unidad de x**.

- *Pista 1:* uno de los dos números te lo dice el enunciado casi con todas las letras.
- *Pista 2:* «pasa por (0, 10)» quiere decir «cuando x vale 0, f(x) vale 10». ¿Cuál de
  las dos perillas controla eso?
- *Pista 3:* «baja» quiere decir que la pendiente es negativa.

In [ ]:
# TODO 1: cambiá estos dos números.
mi_a = 1.0
mi_b = 0.0

fig, ax = g.figura(7, 4.2, titulo="TODO 1", xlabel="x", ylabel="f(x)")
ax.plot(xs, recta(xs, mi_a, mi_b), color=g.MODELO, linewidth=3)
ax.scatter([0], [10], s=110, color=g.OK, zorder=5, label="tiene que pasar por acá")
ax.legend()
plt.show()

revisar(abs(mi_b - 10) < 1e-9,
        "la altura está: cuando x vale 0, tu recta vale 10.",
        "tu recta todavía no pasa por (0, 10). Fijate cuánto da `recta(0, mi_a, mi_b)`: "
        "da exactamente `mi_b`, porque `a * 0` es cero.")
revisar(abs(mi_a + 2) < 1e-9,
        "la inclinación está: baja 2 por cada unidad de x.",
        "«baja 2 por cada unidad» quiere decir que por cada paso de x el valor cambia "
        "en −2. Ese número es `mi_a`, y tiene que ser negativo.")

---

## 3. Los datos de verdad

Ahora dejamos las funciones inventadas y traemos datos.

**Qué son.** Cada año, al abrir las sesiones ordinarias del Congreso, quien preside la
Nación lee un mensaje ante la Asamblea Legislativa: el 1 de mayo hasta 1994, y el
1 de marzo desde 1995 (la reforma constitucional cambió la fecha). Es de los documentos
más comparables de la política argentina: mismo emisor institucional, mismo auditorio,
misma función. Tomamos los discursos de 1984 a 2019 y contamos qué **porcentaje de las
palabras** pertenece a un léxico de 24 palabras sobre *seguridad*.

**Las salvedades, ahora y no al final** (esto es la mitad del oficio):

- **El léxico lo escribimos nosotros.** Está a la vista en `datos/preparar_corpus.py`.
  Si le sacás «policía», la serie cambia. Ninguna lista de palabras es neutral.
- **La palabra central tiene varios sentidos.** «Seguridad» es casi la mitad de los
  aciertos del léxico (228 de 461), y por lo menos 64 de esas 228 son «seguridad
  **social**», «jurídica», «alimentaria» o «vial». O sea que alrededor del 14 % de lo
  que cuenta la serie habla de jubilaciones, contratos, comida o tránsito, no de
  delito. Es el problema de validez más grande de esta medida, y más importante que
  el de «policía». (Medido en `medir-referencias.py`.)
- **Es porcentaje, no conteo**, porque el largo de los discursos varía por un factor
  de nueve (de 2.873 palabras en 1996 a 25.217 en 2013). Contar menciones crudas
  mediría elocuencia, no agenda.
- **El discurso no lo escribe quien lo lee.** Lo escribe un equipo y se negocia con
  los ministerios. Lo que medimos es *la agenda que la Presidencia decidió mostrarle
  al Congreso ese año*. Eso es un objeto de estudio legítimo y valioso. No es «lo que
  el presidente pensaba». Escribí la frase correcta en el paper.
- **Hay dos ceros exactos (2002 y 2009)** y no significan que no hubo inseguridad:
  significan que ese año, con ese léxico, no apareció ninguna de esas 24 palabras.
  Un cero en una serie de proporciones casi siempre es un problema de instrumento,
  no un hallazgo.
- **2006 es una copia de 2005.** En el corpus fuente, el discurso de 2006 es el mismo
  texto que el de 2005 (mismas 18.234 palabras, mismo 0,1645 %). La serie tiene 36
  puntos pero 35 discursos distintos. Hoy lo dejamos a la vista; en la clase 4 vamos
  a ver por qué un duplicado así puede engañar a un modelo.
- **La fecha cambió en el medio.** Los 11 discursos de 1984 a 1994 son de mayo, y los
  de 1995 en adelante, de marzo. Un discurso de mayo ya puede hablar de lo que pasó
  en marzo y abril; uno de marzo, todavía no.

In [ ]:
# El patrón de la casa: la serie se lee de la muestra que viene en el repo
# (datos/muestras/series/, no hace falta red); y si por algo no está, se usa la
# copia de emergencia de acá abajo. Nunca se rompe una clase por falta de internet.
try:
    anios, seguridad = datos.serie_menciones("seguridad", desde=1984, hasta=2019)
    origen = "el corpus del curso"
except Exception as problema:                      # noqa: BLE001
    print("No pude leer el corpus, uso la copia de emergencia:", problema)
    anios = np.arange(1984, 2020, dtype=float)
    seguridad = np.array([
        0.0782, 0.1166, 0.0382, 0.0082, 0.0660, 0.0324, 0.0522, 0.0516,
        0.2441, 0.0765, 0.0578, 0.1120, 0.1044, 0.2250, 0.4357, 0.1468,
        0.4138, 0.3873, 0.0000, 0.0246, 0.1958, 0.1645, 0.1645, 0.1006,
        0.3806, 0.0000, 0.0928, 0.2338, 0.0750, 0.1626, 0.0535, 0.0488,
        0.4267, 0.2496, 0.3112, 0.2453])
    origen = "la copia de emergencia escrita en esta celda"

print(f"Datos ({origen}): {len(anios)} años, de {int(anios[0])} a {int(anios[-1])}.")
print(f"El máximo fue en {int(anios[seguridad.argmax()])}: "
      f"{seguridad.max():.3f}% de las palabras del discurso.")
print(f"El promedio de los 36 años es {seguridad.mean():.3f}%.")

g.plot_serie(anios, seguridad, etiqueta="menciones a seguridad",
             titulo="Cuánto se habló de seguridad en la apertura de sesiones")
plt.show()

> La visualización interactiva de esta parte está en
> `viz/maquina-de-perillas.html`. Abrila en el navegador (funciona sin internet) y
> movele los sliders antes de seguir: lo que viene ahora es lo mismo, en código.

---

## 4. Hacelo vos — el ajuste a ojo

Queremos resumir esos 36 puntos con una recta. Para eso necesitamos dos cosas:

1. **La recta**, que ya la tenemos (`recta`, con sus dos perillas).
2. **Un número que diga qué tan mal está**, para poder comparar dos intentos sin
   discutir de gustos.

Ese número lo armamos así: para cada año, miramos **cuánto le erró** la recta
(la distancia vertical entre el punto y la línea), lo elevamos al cuadrado y
promediamos.

¿Por qué al cuadrado? Por dos razones, y las dos importan:

- para que un error de +0,1 no se **cancele** con uno de −0,1;
- para que los errores grandes **pesen mucho más** que los chicos.

Esa segunda razón es una **decisión metodológica**, no un tecnicismo: estás diciendo
que fallar feo en un año es mucho peor que fallar poquito en varios. En la clase 3 la
discutimos en serio. Hoy la usamos y le decimos **desajuste**.

Un detalle de comodidad: en vez de meterle el año crudo (1998) a la función, le
metemos **los años que pasaron desde 1984** (o sea, 14). Es la misma recta; lo único
que cambia es que así `b` significa «cuánto vale en 1984», que es una frase que se
puede decir en voz alta.

In [ ]:
def desajuste(a, b):
    """El número que dice qué tan mal está la recta. Más chico es mejor."""
    errores = seguridad - recta(anios - 1984, a, b)
    return np.mean(errores ** 2)


def probar(a, b):
    """Dibuja tu recta sobre los datos, marca los errores e imprime el número."""
    d = desajuste(a, b)
    peor = int(anios[np.abs(seguridad - recta(anios - 1984, a, b)).argmax()])

    g.plot_ajuste(anios, seguridad,
                  modelo=lambda t: recta(t - 1984, a, b),
                  loss=d, nombre_loss="desajuste",
                  xlabel="año", ylabel="% de palabras del discurso",
                  etiqueta_modelo=f"mi recta (a={a}, b={b})",
                  titulo="Ajustar a ojo: girar las perillas y mirar el número")
    plt.show()

    print(f"DESAJUSTE = {d:.4f}      (más chico es mejor)")
    print(f"El año peor dibujado es {peor}.")
    print(f"En castellano, tu modelo dice: en 1984 el tema ocupaba {b:.3f}% de las "
          f"palabras, y de ahí {'sube' if a >= 0 else 'baja'} {abs(a):.4f} puntos por año.")
    return d


probar(0.0, 0.10);   # el «;» del final sólo evita que Jupyter repita el número abajo

### La celda de juego (esta se corre veinte veces)

Cambiá los dos números y volvé a correr. **Está bien correrla veinte veces: es
exactamente lo que se espera que hagas.**

Estrategia que funciona (y que tiene nombre, pero se lo ponemos en la clase 3):
mové **una perilla por vez**. Primero mirá si la recta está muy arriba o muy abajo
—eso es `b`— y recién cuando esté a la altura correcta mirá si está muy inclinada
—eso es `a`—.

In [ ]:
probar(a=0.000, b=0.100);    # ← cambiá los dos números y volvé a correr la celda

### `TODO 2` — bajá el desajuste

Arrancamos en `a = 0.000`, `b = 0.100`, que da un desajuste de **0.0195**.
Tu objetivo: **bajarlo a menos de 0.0160**.

- *Pista 1:* mirá el gráfico de arriba. ¿La recta está en general por debajo de los
  puntos o por encima? Eso te dice para dónde mover `b`.
- *Pista 2:* después de acomodar la altura, fijate si los puntos de la derecha quedan
  sistemáticamente arriba de la recta. Si es así, `a` tiene que subir.
- *Pista 3:* los pasos útiles para `a` son chiquitos, del orden de 0.001. Para `b`,
  del orden de 0.01.

In [ ]:
# TODO 2: encontrá dos números que dejen el desajuste por debajo de 0.0160.
mejor_a = 0.000
mejor_b = 0.100

d = probar(mejor_a, mejor_b)

revisar(d < 0.0160,
        f"lo bajaste a {d:.4f}. Anotá esos dos números: los vas a comparar con los "
        f"del resto del aula.",
        f"vas por {d:.4f} y buscamos menos de 0.0160. Probá primero subiendo `mejor_b` "
        f"hasta que la recta quede en el medio de la nube, y recién después tocá `mejor_a`.")

El piso de la recta sobre estos datos está **medido**: es **0.0146**, con
`a = 0.0040` y `b = 0.085` (lo calculó `medir-referencias.py`, barriendo todos los
valores posibles). Con dos perillas no hay nada mejor que eso.

Y acá va el dato incómodo, que es el mejor de la clase de hoy: **una raya horizontal
en el promedio da 0.0165**. O sea que la mejor recta que existe apenas le gana a *no
hacer nada*. Eso no es un fracaso del método: **es un resultado sobre el fenómeno**.
La tendencia lineal en menciones a seguridad, en estos 36 años, es débil de verdad.

### `TODO 3` (opcional, para quien va rápido) — que la busque la computadora

Vos moviste las perillas a mano. ¿Y si probamos 40 valores de `a` de una y nos
quedamos con el mejor?

- *Pista 1:* `np.linspace(-0.01, 0.02, 40)` te da los 40 valores a probar.
- *Pista 2:* guardá el mejor en una variable y comparalo contra cada nuevo.
  Arrancá con `mejor = 999` para que el primero siempre gane.

Esto que estás por escribir se llama **búsqueda por grilla** y es la manera más
tonta de optimizar. Funciona con dos perillas. Con miles de millones, no. Por eso existe
la clase 3.

In [ ]:
b_fijo = 0.085          # dejamos la altura quieta y buscamos sólo la inclinación

mejor_valor = 999.0
mejor_a_grilla = None

for candidata in np.linspace(-0.01, 0.02, 40):
    d = desajuste(candidata, b_fijo)
    # TODO 3: acá va la condición. ¿Cuándo hay que quedarse con esta candidata?
    #         Cambiá `False` por la comparación que corresponda (usá `d` y `mejor_valor`).
    if False:
        mejor_valor = d
        mejor_a_grilla = candidata

if mejor_a_grilla is None:
    print("Todavía no se guardó ninguna candidata: la condición del `if` nunca se cumple.")
else:
    print(f"De 40 candidatas, la mejor inclinación es {mejor_a_grilla:.4f} "
          f"y da un desajuste de {mejor_valor:.4f}.")

revisar(mejor_a_grilla is not None and mejor_valor < 0.0150,
        "la computadora encontró en un parpadeo lo que a vos te llevó un rato. "
        "Y probó siempre lo mismo, sin criterio: por eso no escala.",
        "el `if` tiene que guardar la candidata sólo cuando su desajuste es MÁS CHICO "
        "que el mejor que tenías hasta ese momento. En Python eso se escribe "
        "`d < mejor_valor`.")

---

## 5. El zoo, en código

La recta es una figura entre muchas. Estas son las seis formas de la slide del zoo:
cinco con perillas para jugar, y el **escalón**, que afirma «antes de este umbral, una
cosa; después, otra» y que en el `TODO 4` vas a fabricar vos a partir de la sigmoide.
**No hay nada que memorizar acá**: lo que importa es reconocer la *forma* y saber
qué afirma cada una sobre el mundo.

*Tiempos:* en clase se hace hasta la sección 4. Las secciones 5 y 6 son para después;
la 6 conviene hacerla antes de la clase 2 (son unos diez minutos).

In [ ]:
def parabola(x, a, v):
    """Sube y después baja (o al revés). La U invertida de siempre."""
    return 4 + a * (x - v) ** 2


def exponencial(x, r, c):
    """Crecimiento que se acelera solo."""
    return c * np.exp(r * x)


def sigmoide(x, k, m):
    """La S: empieza lento, se acelera y se satura. Guardala."""
    z = np.clip(-k * (x - m), -60, 60)   # para que con k enormes la computadora no se queje
    return 8 / (1 + np.exp(z))


def seno(x, periodo, amplitud):
    """Todo lo que se repite."""
    return 4 + amplitud * np.sin(2 * np.pi * x / periodo)


zoo = [
    ("recta:  cambio constante",      lambda t: recta(t, 0.55, 1.0)),
    ("parábola:  sube y baja",        lambda t: parabola(t, -0.22, 5.0)),
    ("exponencial:  explota",         lambda t: exponencial(t, 0.32, 0.35)),
    ("sigmoide:  se satura",          lambda t: sigmoide(t, 1.6, 5.0)),
    ("seno:  se repite",              lambda t: seno(t, 5.0, 2.4)),
    ("escalón:  un umbral",           lambda t: np.where(t < 5, 1.0, 7.0)),
]

t = np.linspace(0, 10, 300)
fig, ejes = g.figura(13, 5.2, filas=2, columnas=3, titulo="El zoo de funciones")
for eje, (nombre, f) in zip(ejes.ravel(), zoo):
    eje.plot(t, f(t), color=g.MODELO, linewidth=2.8)
    eje.set_title(nombre, fontsize=10)
    eje.set_ylim(-1, 9.5)
plt.show()

### Celda de juego del zoo

Elegí una y movele las perillas. Sugerencia: empezá por la sigmoide, que es la que
vas a necesitar en el `TODO 4`.

In [ ]:
# ← cambiá estos dos números y volvé a correr
k = 1.0      # qué tan empinado es el salto
m = 5.0      # dónde está el salto (en qué x pasa por la mitad)

fig, ax = g.figura(7, 4.2, titulo=f"sigmoide con k={k}, m={m}",
                   xlabel="x", ylabel="f(x)")
ax.set_ylim(-1, 9.5)
ax.plot(t, sigmoide(t, k, m), color=g.MODELO, linewidth=3)
ax.axhline(8, color=g.VERDAD, linestyle=(0, (5, 4)), linewidth=1.5)
ax.annotate("el techo", (0.3, 8), textcoords="offset points", xytext=(0, 6),
            color=g.VERDAD, fontweight="bold")
plt.show()

### `TODO 4` — hacé que la sigmoide parezca un escalón

Buscá valores de `k` y `m` que hagan que la S se parezca lo más posible a un
**escalón**: plana abajo, salto vertical, plana arriba.

- *Pista:* uno de los dos controla *lo empinado* del salto y el otro sólo lo corre
  de lugar. Exagerá el que controla lo empinado: probá con 50, con 200.
- *Después, la pregunta que importa:* si tu modelo dice que el cambio fue un escalón
  perfecto, ¿qué está afirmando sobre el mundo? ¿Y es creíble?

In [ ]:
# TODO 4: hacé que esto parezca un escalón.
mi_k = 1.0
mi_m = 5.0

fig, ax = g.figura(7, 4.2, titulo="TODO 4: ¿se puede hacer un escalón?",
                   xlabel="x", ylabel="f(x)")
ax.set_ylim(-1, 9.5)
ax.plot(t, sigmoide(t, mi_k, mi_m), color=g.MODELO, linewidth=3, label="tu sigmoide")
ax.plot(t, np.where(t < 5, 0, 8), color=g.VERDAD, linestyle=(0, (5, 4)),
        linewidth=2, label="el escalón perfecto")
ax.legend()
plt.show()

# ¿cuánto le falta a tu sigmoide para ser un escalón?
distancia = float(np.mean((sigmoide(t, mi_k, mi_m) - np.where(t < 5, 0, 8)) ** 2))
print(f"Distancia al escalón: {distancia:.4f}")

revisar(distancia < 0.05,
        "casi un escalón. La perilla que exageraste es la que controla lo empinado. "
        "En teoría la distancia nunca llega a cero: por más que la aprietes, la sigmoide "
        "sigue siendo suave. (Si ves 0.0000 es porque la diferencia quedó más chica que "
        "lo que se imprime: medimos en 300 puntos y ninguno cae justo en el salto.) "
        "Esa suavidad importa: en la clase 3 vas a ver que para aprender hace falta "
        "pendiente, y el escalón no tiene.",
        f"vas por {distancia:.4f} y buscamos menos de 0.05. Probá subiendo mucho una "
        f"de las dos —mucho quiere decir 50, o 200— y fijate cuál de las dos cambia la "
        f"FORMA en vez de correr la curva de lugar. Y el salto tiene que quedar en x = 5.")

---

## 6. Ejercicio — la serie que rompe la recta

**Consigna.** Repetí todo lo de arriba, pero con las menciones a **inflación** en
vez de seguridad. Ajustá la mejor recta que puedas y contestá, en una oración, por
qué la recta no alcanza acá.

**Cómo saber si te salió.** No hay un número que tengas que alcanzar. El criterio es
otro: **tenés que poder decir en voz alta qué le pasa a la recta y por qué**.
La celda arranca con una recta lejos del mejor lugar (desajuste 0.00231) y te
muestra cinco decimales, para que veas cuánto se puede bajar y dónde está el piso.
Si tu respuesta es «porque los datos son ruidosos», mirá el gráfico otra vez: hay
algo más específico.

*(Esa respuesta es, literalmente, el gancho de la clase 2.)*

In [ ]:
try:
    anios_inf, inflacion = datos.serie_menciones("inflacion", desde=1984, hasta=2019,
                                                 suavizada=True)
except Exception:                                  # noqa: BLE001
    anios_inf = np.arange(1984, 2020, dtype=float)
    inflacion = np.array([
        0.1143, 0.1016, 0.0588, 0.0658, 0.0674, 0.1152, 0.0921, 0.0825,
        0.0601, 0.0719, 0.1105, 0.0966, 0.0745, 0.0359, 0.0365, 0.0297,
        0.0211, 0.0219, 0.0465, 0.0410, 0.0555, 0.0583, 0.0850, 0.0807,
        0.0532, 0.0323, 0.0217, 0.0320, 0.0267, 0.0321, 0.0340, 0.1143,
        0.1440, 0.1837, 0.1321, 0.1268])

# Usamos la serie SUAVIZADA (media móvil de 3 años) para que se vea la forma.
# Ojo: suavizar cambia la pregunta. La serie cruda contesta «cuánto se habló de esto
# ESE año»; la suavizada contesta «en qué parte del período el tema estaba en agenda».
# Las dos son legítimas; usar una y reportar la otra, no.

def desajuste_inf(a, b):
    return float(np.mean((inflacion - recta(anios_inf - 1984, a, b)) ** 2))


# TODO: cambiá estos dos números buscando el mejor ajuste posible.
a_inf = -0.002
b_inf = 0.120

g.plot_ajuste(anios_inf, inflacion,
              modelo=lambda t: recta(t - 1984, a_inf, b_inf),
              loss=desajuste_inf(a_inf, b_inf), nombre_loss="desajuste",
              xlabel="año", ylabel="% de palabras del discurso",
              etiqueta_modelo=f"mi recta (a={a_inf}, b={b_inf})",
              titulo="Menciones a «inflación» (media móvil de 3 años)")
plt.show()

print(f"Tu desajuste: {desajuste_inf(a_inf, b_inf):.5f}")
print("La MEJOR recta posible da 0.00156 (medido: a = 0.00026, b = 0.0665).")
print("Una raya horizontal en el promedio da 0.00157 (medido).")
print("O sea: empatan hasta el cuarto decimal.")
print("La mejor recta que existe casi no le gana a no hacer absolutamente nada.")

**Tu respuesta** *(escribí acá, en una oración: por qué la recta no alcanza)*

_(doble clic en esta celda para escribir)_

---

Una ayuda para pensarla, si no sale: mirá la forma de la nube. Sube, baja, y vuelve
a subir. ¿Cuántas veces cambia de dirección? ¿Cuántas veces puede cambiar de
dirección una recta?

---

## 7. Qué te llevás

**Las tres cosas de hoy:**

1. **Una función es una máquina con perillas.** Entra algo, sale algo, y las perillas
   eligen qué figura dibuja. Todo el curso es esta imagen, con máquinas cada vez más
   grandes.
2. **Variable y parámetro son cosas distintas** aunque los dos sean números. La regla
   del dedo: ¿cambia de una fila a la otra de mi base? Es variable. ¿Es el mismo para
   toda la base? Es parámetro.
3. **Elegir la forma es una decisión teórica tuya, no del modelo.** La máquina elige
   el miembro de la familia; la familia la elegís vos. Y hoy viste que hay figuras
   que ninguna familia simple puede dibujar.

**El vocabulario nuevo** (también está en `recursos/glosario.md`):

| término | en castellano llano |
|---|---|
| *función* | una máquina: le metés un número, sale otro |
| *variable* | lo que entra; cambia de un caso a otro |
| *parámetro* | la perilla; es la misma para toda la base |
| *ajustar* (*to fit*) | mover las perillas hasta que la figura se parezca a los datos |
| *optimización* | mover perillas para minimizar un número |
| *desajuste* | el número que dice qué tan mal está tu dibujo (en la clase 3 tiene nombre propio) |

**En tu propia investigación, desde ya.**

1. Tomá una serie tuya (una variable por año, por mes, por provincia).
2. **Graficala.** Puntos, no barras. Mirala un rato antes de modelar nada: la mitad
   de los errores metodológicos de un paper se evitan graficando primero.
3. Ajustale una recta a ojo con las celdas de acá arriba y escribí, **en castellano
   sustantivo**, qué significa cada parámetro. Si tu pendiente da 0.4, no escribas
   «la pendiente es 0.4»: escribí «cada año sube 0.4 puntos». Esa traducción es tu
   trabajo, no el del modelo.

**Lo que queda prometido.**

- Hoy dijimos «desajuste» y no dijimos el nombre técnico. **Se paga en la clase 3.**
- Hoy nos chocamos con figuras que ninguna forma simple puede dibujar.
  **Se paga en la clase 2, y la salida no es una fórmula más rara: es apilar.**
- Y guardá la sigmoide. Vuelve en la clase 2, y en la 9 está adentro de la regresión
  logística aunque no la veamos.

**Para antes de la clase 2** (menos de una hora en total): el cuestionario
(`cuestionario.md`, unos 20 minutos) y la sección 6 de este notebook, si no la
hiciste. La sección 5 y el bonus del cuestionario son opcionales.

---

### Para seguir

Las referencias de la clase están en `referencias.md`. Empezá por las de ciencias
sociales: lo que importa no es el paper de ML, es el trabajo que muestra qué se
puede y qué no se puede afirmar con esto.